# Treinamento distribuído com PySpark + ClickHouse (Railway)

Os CSVs do dataset CICDDoS2019 foram carregados em tabelas `MergeTree` numa instância
ClickHouse hospedada na Railway (ver `scripts/import_raw_csvs.sh`). São **~60 milhões de
fluxos de rede** em 18 tabelas — os arquivos somam **~21 GB** e, expandidos em memória
como DataFrame, não cabem de uma vez para treinamento em batch.

Este notebook mostra o caminho usado quando o dado **não cabe na memória**:

1. conecta o **PySpark** ao ClickHouse via o conector oficial
   (`com.clickhouse.spark:clickhouse-spark-runtime-3.5_2.12`) usando HTTPS/SSL;
2. registra o ClickHouse como catálogo Spark (`clickhouse.<database>.<tabela>`);
3. lê as tabelas como **DataFrames particionados** — o Spark busca os dados em partições
   paralelas e só mantém em memória o que cada tarefa processa (o driver nunca
   materializa o dataset inteiro);
4. roda um **exemplo simples de treinamento distribuído** com MLlib (regressão logística
   multinomial) classificando a família de ataque de cada fluxo.

> As credenciais ficam em `.env` na raiz do repositório (não versionado). Este notebook
> só faz leitura no ClickHouse — não altera nenhuma tabela.

## 0. Ambiente

Pré-requisitos: Python 3.10+, Java 17 ou 21 no `PATH` e as dependências do
[`requirements.txt`](../requirements.txt):

```bash
cd clickhouse-railway
python3 -m venv .venv
.venv/bin/pip install -r requirements.txt
```

Ajustes opcionais (cluster, memória) — exporte antes de iniciar o Jupyter:

```bash
export SPARK_MASTER="local[*]"            # ou spark://host:7077, yarn, k8s://...
export SPARK_DRIVER_MEMORY=6g             # memória do driver (local[*] = JVM única)
export SPARK_SHUFFLE_PARTITIONS=24
```

In [ ]:
import os
from pathlib import Path

def _parse_env_line(line: str):
    line = line.strip()
    if not line or line.startswith("#") or "=" not in line:
        return None
    k, _, v = line.partition("=")
    return k.strip(), v.strip().strip('"').strip("'")

def load_env(path):
    env = {}
    p = Path(path)
    if p.exists():
        for line in p.read_text().splitlines():
            kv = _parse_env_line(line)
            if kv:
                env[kv[0]] = kv[1]
    return env

def repo_root():
    # Notebook roda em <repo>/notebooks; .env fica em <repo>
    here = Path.cwd()
    for cand in (here, here.parent, here.parent.parent):
        if (cand / ".env").is_file() and (cand / "scripts").is_dir():
            return cand
    raise FileNotFoundError("Não achei o .env / a estrutura do repositório")

ROOT = repo_root()
ENV = load_env(ROOT / ".env")
print("Raiz do repositório:", ROOT)
print("Variáveis carregadas:", sorted(ENV))

## 1. SparkSession + catálogo ClickHouse

Registramos o ClickHouse como catálogo Spark (`clickhouse`) usando o
`ClickHouseCatalog` do conector oficial. A instância da Railway é **HTTPS na porta 443**,
então configuramos `protocol=https`, `http_port=443` e `option.ssl=true`.

O conector e o cliente Java do ClickHouse são baixados do Maven Central na primeira
execução (`spark.jars.packages`) e ficam cacheados em `.ivy2/` dentro do projeto
(ignorado pelo Git). Num ambiente normal você pode remover a linha do `spark.jars.ivy`
para usar o cache padrão `~/.ivy2`.

In [ ]:
from pyspark.sql import SparkSession

CLICKHOUSE = {
    "host": ENV["CLICKHOUSE_HOST"],
    "port": ENV.get("CLICKHOUSE_PORT", "443"),
    "user": ENV["CLICKHOUSE_USER"],
    "password": ENV["CLICKHOUSE_PASSWORD"],
    "database": ENV["CLICKHOUSE_DATABASE"],
}

# Artefatos Maven do conector oficial p/ Spark 3.5 (ver
# https://github.com/ClickHouse/spark-clickhouse-connector). O Spark não aceita o
# classifier ":all" em --packages; o clickhouse-jdbc simples resolve as dependências
# transitivas (http-client etc.) pelo POM.
CH_PACKAGES = (
    "com.clickhouse.spark:clickhouse-spark-runtime-3.5_2.12:0.8.1,"
    "com.clickhouse:clickhouse-jdbc:0.6.3"
)
IVY_DIR = os.environ.get("SPARK_IVY_DIR") or str(ROOT / ".ivy2")

builder = (
    SparkSession.builder
    .appName("clickhouse-railway-training")
    .master(os.environ.get("SPARK_MASTER", "local[*]"))
    .config("spark.jars.packages", CH_PACKAGES)
    .config("spark.jars.ivy", IVY_DIR)
    .config("spark.driver.memory", os.environ.get("SPARK_DRIVER_MEMORY", "6g"))
    .config("spark.sql.shuffle.partitions", os.environ.get("SPARK_SHUFFLE_PARTITIONS", "24"))
    # ---- catálogo "clickhouse" (DataSourceV2) -------------------------------
    .config("spark.sql.catalog.clickhouse", "com.clickhouse.spark.ClickHouseCatalog")
    .config("spark.sql.catalog.clickhouse.host", CLICKHOUSE["host"])
    .config("spark.sql.catalog.clickhouse.protocol", "https")
    .config("spark.sql.catalog.clickhouse.http_port", CLICKHOUSE["port"])
    .config("spark.sql.catalog.clickhouse.user", CLICKHOUSE["user"])
    .config("spark.sql.catalog.clickhouse.password", CLICKHOUSE["password"])
    .config("spark.sql.catalog.clickhouse.database", CLICKHOUSE["database"])
    .config("spark.sql.catalog.clickhouse.option.ssl", "true")
)

spark = builder.getOrCreate()
print("Spark", spark.version, "| master:", spark.sparkContext.master)
print("Paralelismo default (workers/threads):", spark.sparkContext.defaultParallelism)

## 2. Prova de conexão: leitura particionada

`clickhouse.<database>.<tabela>` vira um nome de tabela normal do Spark:
`spark.read.table(...)` dispara uma leitura **particionada** do ClickHouse — o dataset
nunca é materializado inteiro no driver.

In [ ]:
from pyspark.sql import functions as F

DB = CLICKHOUSE["database"]

def table(name: str):
    return spark.read.table(f"clickhouse.{DB}.{name}")

# tabela pequena só para provar conectividade (191.694 fluxos Portmap)
portmap = table("data2_portmap")
print("partições da leitura:", portmap.rdd.getNumPartitions())

portmap.select(
    "unnamed_0", "flow_id", "source_ip", "source_port",
    "destination_ip", "destination_port", "protocol", "timestamp", "label"
).show(5, truncate=False)

### 2.1 As tabelas

As 18 tabelas têm o **mesmo schema de 88 colunas** (cabeçalho CICDDoS2019), importadas
cruas como `String` e compactadas com ZSTD no ClickHouse. A carga foi validada
comparando `count()` no ClickHouse com `wc -l` dos CSVs de origem
(esperado = linhas_do_arquivo − 1) — todas as 18 bateram, e amostras de linhas conferem
valor a valor com os arquivos. A coluna `Label` identifica a família do ataque
(`Syn`, `DrDoS_NTP`, `UDP-lag`, `BENIGN`, ...).

| Tabela (ClickHouse) | Origem (CSV) | Linhas |
|---|---|---|
| `data_syn` | `data/Syn.csv` | 1.582.681 |
| `data_udplag` | `data/UDPLag.csv` | 370.605 |
| `data_drdos_ntp` | `data/DrDoS_NTP.csv` | 1.217.007 |
| `data2_portmap` | `data2/Portmap.csv` | 191.694 |
| `data2_udplag` | `data2/UDPLag.csv` | 725.165 |
| `data2_mssql` | `data2/MSSQL.csv` | 5.775.786 |
| `data_tftp` | `data/TFTP.csv` | 20.107.827 |
| `data_drdos_dns` | `data/DrDoS_DNS.csv` | 5.074.413 |
| ... | ... | ... |

O mapa completo CSV → tabela está em `scripts/import_raw_csvs.sh`.

## 3. Exemplo simples de treinamento distribuído

**Objetivo didático**: classificar a **família de ataque** do fluxo usando as colunas
numéricas de estatísticas de fluxo (tamanhos de pacote, contagens de flags, IAT etc.).
Treinamos uma regressão logística multinomial do MLlib, que roda de forma
**distribuída/iterativa** sobre as partições — por isso o dataset pode ser bem maior que
a memória do driver.

Aqui juntamos 3 tabelas por família (`Syn`, `UDP-lag`, `DrDoS_NTP`) e limitamos cada
classe a `LIMIT_PER_CLASS` linhas — troque por `None` para usar as tabelas inteiras.
Colunas de identificação/texto (`flow_id`, IPs, timestamps) são descartadas.

In [ ]:
# família de ataque -> tabela que a contém (o Label dentro da tabela é o da família)
FAMILIES = {
    "data_syn":       "Syn",
    "data_udplag":    "UDP-lag",
    "data_drdos_ntp": "DrDoS_NTP",
}

LIMIT_PER_CLASS = 300_000   # demo rápida; None = tabela inteira
SEED = 42

# colunas de identificação/texto — não entram como features
DROP_COLS = ["unnamed_0", "flow_id", "source_ip", "destination_ip", "timestamp"]

frames = []
for tbl, fam in FAMILIES.items():
    part = (
        table(tbl)
        .where(F.col("label") == fam)
        .drop(*DROP_COLS)
    )
    if LIMIT_PER_CLASS is not None:
        part = part.limit(LIMIT_PER_CLASS)
    frames.append(part)

data = frames[0]
for f in frames[1:]:
    data = data.unionByName(f)

print("classes:", {tbl: fam for tbl, fam in FAMILIES.items()})
print("linhas amostradas:", data.count())
print("partições após a leitura:", data.rdd.getNumPartitions())

### 3.1 Pré-processamento distribuído

Tudo chega como `String`. Convertemos as features para `double` e tratamos valores
não-finites (`NaN`/`Infinity` — comuns quando o dataset original divide por zero) como 0.
Esse passo também roda distribuído.

In [ ]:
NUMERIC_COLS = [c for c in data.columns if c != "label"]

def to_numeric_finite(df, cols):
    # String -> double; NaN/Inf -> NULL; NULL -> 0.0
    for c in cols:
        df = df.withColumn(c, F.col(c).cast("double"))
        df = df.withColumn(
            c,
            F.when(
                F.isnan(F.col(c)) | (F.col(c) == float("inf")) | (F.col(c) == float("-inf")),
                None,
            ).otherwise(F.col(c)),
        )
    return df.na.fill(0.0, subset=cols)

data = to_numeric_finite(data, NUMERIC_COLS)
data = data.select(*NUMERIC_COLS, "label")

print("linhas após limpeza:", data.count(), "| features:", len(NUMERIC_COLS))
data.cache()
print("partições para o treinamento:", data.rdd.getNumPartitions())

### 3.2 Treino distribuído (MLlib)

`StringIndexer` transforma o `Label` em índice; `VectorAssembler` empacota as features;
a `LogisticRegression` multinomial treina um modelo por classe com otimização
distribuída (LBFGS). O `fit` itera sobre as partições trocando só vetores pequenos de
gradiente entre os workers.

In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.feature import StringIndexer, VectorAssembler

train, test = data.randomSplit([0.8, 0.2], seed=SEED)
print("train:", train.count(), "| test:", test.count())

indexer = StringIndexer(inputCol="label", outputCol="label_idx")
assembler = VectorAssembler(inputCols=NUMERIC_COLS, outputCol="features")
lr = LogisticRegression(
    featuresCol="features",
    labelCol="label_idx",
    maxIter=25,
    regParam=0.01,
    family="multinomial",
)

pipeline = Pipeline(stages=[indexer, assembler, lr])
model = pipeline.fit(train)

classes = model.stages[0].labels
print("mapeamento índice -> classe:", {i: c for i, c in enumerate(classes)})

In [ ]:
pred = model.transform(test)

ev = MulticlassClassificationEvaluator(labelCol="label_idx", predictionCol="prediction")
accuracy = ev.evaluate(pred, {ev.metricName: "accuracy"})
f1 = ev.evaluate(pred, {ev.metricName: "f1"})
print(f"accuracy = {accuracy:.4f}")
print(f"f1 (macro) = {f1:.4f}")

# Matriz de confusão (linhas = rótulo real, colunas = predito)
from pyspark.mllib.evaluation import MulticlassMetrics

metrics = MulticlassMetrics(pred.select("prediction", "label_idx").rdd.map(tuple))
cm = metrics.confusionMatrix().toArray()
print("classes:", classes)
print("confusion matrix (real x predito):")
print(cm)

### 3.3 Por que isso não estoura a memória?

* **Leitura particionada** — o conector lê o ClickHouse em partições paralelas; cada
  executor processa um pedaço de cada vez.
* **Treinamento distribuído** — o MLlib agrega gradientes por partição e só troca vetores
  pequenos entre os workers a cada iteração.
* **Lazy evaluation** — `count()`, `show()` e `fit()`/`transform()` disparam jobs; nada é
  materializado no driver sem você pedir (`collect()`).

Para treinar com o dataset completo (18 tabelas, ~60 M de fluxos): remova os
`LIMIT_PER_CLASS`, adicione mais famílias ao dicionário `FAMILIES` e aumente o cluster
(`SPARK_MASTER`, memória de executor). Em `local[*]` o limite é a JVM do driver; num
cluster real cada executor tem a sua própria.

In [ ]:
# Salvar o modelo treinado p/ uso posterior (ex.: servir com Spark ou exportar p/ MLflow):
# model.write().overwrite().save(str(ROOT / "models" / "lr_attack_family"))

# Encerrar a sessão quando terminar (no Jupyter você pode deixar de pé p/ explorar):
# spark.stop()
print("Fim do exemplo.")